# Healthcare Analytics for Doctor Visits
### A Beginner-Friendly Data Analysis Project

This notebook explores a healthcare survey dataset to understand who visits the doctor, how often, and why.

**How this notebook is organized:**
- Code cells here are kept clean and runnable, with only short comments.
- Full plain-language explanations of *why* each step matters live in `docs/EXPLANATIONS.md` — open it side by side with this notebook if you're new to data analysis.

**Dataset:** `data/doctor_visits.csv` — a health survey of 5,190 people, recording doctor visits, age, gender, income, illnesses, chronic conditions, and insurance status.


## 1. Setup
Import the libraries used throughout this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
PALETTE = ["#2F6F6B", "#E0A458", "#8A4F4B", "#5B7C99", "#B7C4C0"]
sns.set_palette(PALETTE)
%matplotlib inline


## 2. Dataset Overview
Load the CSV file and take a first look at its shape and columns.

In [ ]:
df = pd.read_csv("../data/doctor_visits.csv", index_col=0)
print("Rows, columns:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Human-readable versions of scaled columns
df["age_years"] = (df["age"] * 100).round().astype(int)
df["income_actual"] = (df["income"] * 10000).round(0)
df[["age", "age_years", "income", "income_actual"]].head()


## 3. Missing Values
Check whether any column has missing (empty) entries.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})


## 4. Duplicate Rows
Check for duplicate records in the dataset.

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates} out of {len(df)} ({n_duplicates/len(df)*100:.1f}%)")


In [ ]:
# Preview a few duplicated rows
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)


## 5. Data Types & Cleaning
Confirm each column has an appropriate data type, and standardize categorical text columns.

In [ ]:
df.dtypes


In [ ]:
categorical_cols = ["gender", "private", "freepoor", "freerepat", "nchronic", "lchronic"]
for col in categorical_cols:
    df[col] = df[col].str.strip().str.lower()

df[categorical_cols].nunique()


## 6. Basic Statistics
Summary statistics for the numeric columns.

In [ ]:
df[["visits", "age_years", "income_actual", "illness", "reduced", "health"]].describe().round(2)


## 7. Age and Gender Analysis
How are patients distributed by age and gender, and does gender relate to doctor visits?

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.histplot(df["age_years"], bins=20, kde=True, color=PALETTE[0])
plt.title("Age Distribution of Patients")
plt.xlabel("Age (years)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.countplot(x="gender", data=df, color=PALETTE[1])
plt.title("Gender Breakdown")
plt.xlabel("")
plt.ylabel("Number of People")
plt.tight_layout()
plt.show()


In [ ]:
avg_visits_by_gender = df.groupby("gender")["visits"].mean().round(2)
avg_visits_by_gender


In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.barplot(x="gender", y="visits", data=df, color=PALETTE[2], errorbar=None)
plt.title("Average Doctor Visits by Gender")
plt.xlabel("")
plt.ylabel("Average Visits")
plt.tight_layout()
plt.show()


## 8. Illness-Related Analysis
How many illnesses do people report, and how does that relate to doctor visits and general health?

In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.countplot(x="illness", data=df, color=PALETTE[3])
plt.title("Number of Illnesses Reported (Past 2 Weeks)")
plt.xlabel("Illness Count")
plt.ylabel("Number of People")
plt.tight_layout()
plt.show()


In [ ]:
chronic_summary = pd.DataFrame({
    "non_limiting_chronic_%": [(df["nchronic"] == "yes").mean() * 100],
    "limiting_chronic_%": [(df["lchronic"] == "yes").mean() * 100],
}).round(1)
chronic_summary


In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.boxplot(x="illness", y="visits", data=df, color=PALETTE[4])
plt.title("Doctor Visits by Number of Illnesses")
plt.xlabel("Illness Count")
plt.ylabel("Doctor Visits")
plt.tight_layout()
plt.show()


## 9. Doctor Visit Analysis
A closer look at the target variable: how many times people visited the doctor in the past two weeks.

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.countplot(x="visits", data=df, color=PALETTE[0])
plt.title("Number of Doctor Visits (Past 2 Weeks)")
plt.xlabel("Doctor Visits")
plt.ylabel("Number of People")
plt.tight_layout()
plt.show()


In [ ]:
pct_zero_visits = (df["visits"] == 0).mean() * 100
print(f"People with zero doctor visits: {pct_zero_visits:.1f}%")
print(f"Average visits per person: {df['visits'].mean():.2f}")
print(f"Maximum visits recorded: {df['visits'].max()}")


In [ ]:
insurance_visits = df.groupby("private")["visits"].mean().round(2)
insurance_visits


In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.barplot(x="private", y="visits", data=df, color=PALETTE[1], errorbar=None)
plt.title("Average Visits: Private Insurance vs None")
plt.xlabel("Has Private Insurance")
plt.ylabel("Average Visits")
plt.tight_layout()
plt.show()


## 10. Relationships Between Variables
Explore how numeric variables move together using correlation.

In [ ]:
num_cols = ["visits", "age_years", "income_actual", "illness", "reduced", "health"]
corr = df[num_cols].corr().round(2)
corr


In [ ]:
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="crest", square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4.5))
sns.stripplot(x="visits", y="age_years", data=df, alpha=0.25, color=PALETTE[0], jitter=0.3)
plt.title("Age vs Doctor Visits")
plt.xlabel("Doctor Visits")
plt.ylabel("Age (years)")
plt.tight_layout()
plt.show()


In [ ]:
limiting_chronic_visits = df.groupby("lchronic")["visits"].mean().round(2)
limiting_chronic_visits


In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.barplot(x="lchronic", y="visits", data=df, color=PALETTE[2], errorbar=None)
plt.title("Average Visits: Limiting Chronic Condition")
plt.xlabel("Has a Limiting Chronic Condition")
plt.ylabel("Average Visits")
plt.tight_layout()
plt.show()


## 11. Key Insights

- About **80% of people had zero doctor visits** in the two-week survey window — visits are rare and heavily skewed toward zero.
- People with **more reported illnesses** tend to have **more doctor visits** (weak-to-moderate positive relationship).
- A **worse general health score** is also associated with **more doctor visits**.
- People with a **limiting chronic condition** visit the doctor **more than twice as often on average** as those without one.
- **Age** and **private insurance status** show only a weak relationship with visit frequency in this dataset.
- **Gender** shows a small difference: women report slightly more visits on average than men.


## 12. Conclusion

This exploratory analysis shows that doctor visits in this survey are driven more by **health condition indicators** (illness count, general health score, and chronic conditions) than by demographic factors like age or insurance status alone. Because visit counts are rare-event, skewed data, further analysis (for example, a Poisson or negative binomial regression) would be a natural next step for beginners moving beyond exploratory analysis.

See `docs/EXPLANATIONS.md` for a plain-language walkthrough of every section above, and `README.md` for how to set up and run this project.
